# elbo-loss-sum-with-beta — ex2: contrast beta in {0, 1, 4} on per-sample ELBO loss

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `elbo-loss-sum-with-beta`. Running the final beacon cell reports progress against the `VAE: ELBO loss sum with beta` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `VAE: ELBO loss sum with beta` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`elbo-loss-sum-with-beta`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "elbo-loss-sum-with-beta"
DD_SUBTOPIC = "VAE: ELBO loss sum with beta"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## ELBO with beta — three regimes on the same batch

Ex1 computed `loss = recon + beta * kl`. The deepening move is to FEEL
what `beta` does by contrasting three regimes on identical data:

| beta | name        | behaviour                                       |
| ---- | ----------- | ----------------------------------------------- |
| 0    | pure recon  | autoencoder — latent unconstrained, perfect copy |
| 1    | standard ELBO | balance: latent regularized to N(0, I)         |
| 4    | β-VAE       | KL dominates — latent collapses, recon worsens  |

```python
recon_per_sample = ((x_hat - x) ** 2).flatten(1).sum(dim=1)    # (B,)
kl_per_sample    = -0.5 * (1 + 2*logsigma - mu**2 - (2*logsigma).exp()).sum(dim=1)  # (B,)
loss_per_sample  = recon_per_sample + beta * kl_per_sample     # (B,)
```

**Why per-sample, not scalar.** Returning the `(B,)` vector lets you
inspect WHICH samples the model is bad at, AND lets downstream code
decide whether to `.mean()` or `.sum()` for the gradient step.

**Invariant:** at fixed recon + KL, `loss(beta=4) >= loss(beta=1) >=
loss(beta=0)` for every sample (KL is non-negative). That ordering IS
the test.

### Exercise 2 — contrast beta in {0, 1, 4} on per-sample ELBO loss

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze how the beta weight on the KL term affects per-sample ELBO loss by computing `recon + beta * kl` for `beta in {0, 1, 4}` on the same (x, x_hat, mu, logsigma) batch and verifying the non-decreasing-in-beta invariant per sample.
> Keywords: beta-vae, elbo, per-sample, kl-weight
> ```

**KCs targeted:** `per-sample-elbo-decomposition`, `beta-monotone-in-loss-when-kl-positive`

Implement `ex2_elbo_three_betas(x, x_hat, mu, logsigma)`.

Inputs (all batched, same B):
- `x:        (B, D)` — target.
- `x_hat:    (B, D)` — VAE reconstruction.
- `mu:       (B, latent)`
- `logsigma: (B, latent)`

Compute these per-sample tensors (each shape `(B,)`):

1. `recon = ((x_hat - x) ** 2).flatten(1).sum(dim=1)` — squared-error reconstruction, summed over feature dims.
2. `kl    = -0.5 * (1 + 2*logsigma - mu**2 - (2*logsigma).exp()).sum(dim=1)` — diagonal Gaussian KL vs N(0, I), per-sample.
3. For each `beta in [0.0, 1.0, 4.0]`: `loss_beta = recon + beta * kl`.

Return a dict:
```
{
    'recon':      recon,        # (B,)
    'kl':         kl,           # (B,)
    'loss_beta0': loss_at_beta0,  # (B,)
    'loss_beta1': loss_at_beta1,  # (B,)
    'loss_beta4': loss_at_beta4,  # (B,)
}
```

**No `.mean()` at the end.** Keep the per-sample vector.

In [ ]:
def ex2_elbo_three_betas(x: Tensor, x_hat: Tensor, mu: Tensor, logsigma: Tensor) -> dict:
    """Per-sample ELBO loss at beta in {0, 1, 4}."""
    raise NotImplementedError()


def _test_ex2():
    B, D, L = 8, 16, 4
    x        = t.randn(B, D)
    x_hat    = x + 0.1 * t.randn(B, D)              # near-reconstruction
    mu       = t.randn(B, L) * 0.5
    logsigma = t.randn(B, L) * 0.3

    out = ex2_elbo_three_betas(x, x_hat, mu, logsigma)

    # === All keys present, all (B,) ===
    for k in ['recon', 'kl', 'loss_beta0', 'loss_beta1', 'loss_beta4']:
        assert k in out, f'missing key {k!r}; got {list(out.keys())}'
        v = out[k]
        assert isinstance(v, Tensor), f'{k} must be a Tensor, got {type(v).__name__}'
        assert v.shape == (B,), f'{k} must be (B,), got {tuple(v.shape)}'

    # === recon matches the spec exactly ===
    expected_recon = ((x_hat - x) ** 2).flatten(1).sum(dim=1)
    assert t.allclose(out['recon'], expected_recon, atol=1e-5), 'recon formula mismatch'

    # === kl matches the standard diagonal-Gaussian closed form ===
    expected_kl = -0.5 * (1 + 2*logsigma - mu**2 - (2*logsigma).exp()).sum(dim=1)
    assert t.allclose(out['kl'], expected_kl, atol=1e-5), 'kl formula mismatch'

    # === KL is non-negative per sample (Gaussian KL invariant) ===
    assert (out['kl'] >= -1e-5).all(), f'kl must be >= 0 per sample; got min={out["kl"].min():.6f}'

    # === beta=0 ignores KL entirely; loss == recon ===
    assert t.allclose(out['loss_beta0'], out['recon'], atol=1e-5), 'loss at beta=0 must equal recon (KL dropped)'

    # === beta=1 standard ELBO ===
    assert t.allclose(out['loss_beta1'], out['recon'] + out['kl'], atol=1e-5)

    # === beta=4 = recon + 4*kl ===
    assert t.allclose(out['loss_beta4'], out['recon'] + 4.0 * out['kl'], atol=1e-5)

    # === Per-sample monotonicity: when KL > 0, loss is strictly increasing in beta ===
    kl = out['kl']
    pos = kl > 1e-6
    assert pos.any(), 'test fixture needs at least some samples with positive KL'
    assert (out['loss_beta1'][pos] > out['loss_beta0'][pos]).all(), 'loss_beta1 > loss_beta0 where kl>0'
    assert (out['loss_beta4'][pos] > out['loss_beta1'][pos]).all(), 'loss_beta4 > loss_beta1 where kl>0'

    # === Sanity: at mu=0, logsigma=0 (perfect prior match) → KL == 0 ===
    mu0 = t.zeros(B, L)
    ls0 = t.zeros(B, L)
    out2 = ex2_elbo_three_betas(x, x_hat, mu0, ls0)
    assert t.allclose(out2['kl'], t.zeros(B), atol=1e-5), f'mu=0, logsigma=0 must give kl=0; got max={out2["kl"].max():.6f}'
    # When KL=0, all three losses agree exactly.
    assert t.allclose(out2['loss_beta0'], out2['loss_beta4'], atol=1e-5), 'KL=0 ⇒ beta has no effect'

    # === Higher-dim x (image-shaped) still flattens correctly ===
    x_img      = t.randn(B, 3, 4, 4)
    x_hat_img  = x_img + 0.05 * t.randn(B, 3, 4, 4)
    out3 = ex2_elbo_three_betas(x_img, x_hat_img, mu, logsigma)
    expected = ((x_hat_img - x_img) ** 2).flatten(1).sum(dim=1)
    assert t.allclose(out3['recon'], expected, atol=1e-5), 'recon must flatten image dims (B, C, H, W) → (B, C*H*W) before sum'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_elbo_three_betas(x, x_hat, mu, logsigma):
    recon = ((x_hat - x) ** 2).flatten(1).sum(dim=1)
    kl    = -0.5 * (1 + 2 * logsigma - mu ** 2 - (2 * logsigma).exp()).sum(dim=1)
    return {
        'recon':      recon,
        'kl':         kl,
        'loss_beta0': recon + 0.0 * kl,
        'loss_beta1': recon + 1.0 * kl,
        'loss_beta4': recon + 4.0 * kl,
    }
```

**Beta is monotone WHEN KL is positive.** Since KL >= 0 for any diagonal Gaussian, `loss(beta=b)` is non-decreasing in `b` per sample. Strict inequality requires strictly positive KL — at the prior `mu=0, logsigma=0`, all betas tie.

**Why per-sample, not pre-reduced.** Returning `(B,)` lets the caller `.mean()` for the training step OR pick the worst-recon samples for inspection. Pre-reducing throws information away.

**`flatten(1)` over `view(B, -1)`.** Equivalent but `flatten(1)` doesn't require knowing the batch size — useful when the batch dimension comes through a DataLoader with varying batch size at the tail.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()